In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

import os

In [ ]:
import os
import pandas as pd
import numpy as np

base_model_dir = "/content/drive/MyDrive/@antigen_epitope/abdb_v4"
new_base_model_dir = "/content/drive/MyDrive/@antigen_epitope/abdb_new_method/"

all_patch_summaries = []

for fold in range(1,6):
    input_file_path = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/', 'cnn_positive_patch_3points.txt')

    filtered_data_path = os.path.join(new_base_model_dir, f'fold_{fold}/data/', 'all_data_with_SI.csv')
    filtered_combined_df = pd.read_csv(filtered_data_path)
    filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

    protein_names = []
    patch_indices = []

    with open(input_file_path, 'r') as file:
        lines = file.readlines()
        current_protein_name = None
        for line in lines:
            line = line.strip()
            if line.startswith('Patches for protein'):
                current_protein_name = line.split(' ')[3][:-1]
            elif line.startswith('indices:'):
                indices = line.split('[')[1].split(']')[0].split(', ')
                indices = [int(idx) for idx in indices]
                protein_names.append(current_protein_name)
                patch_indices.append(indices)

    for protein_name, indices in zip(protein_names, patch_indices):
        protein_data = filtered_combined_df[filtered_combined_df['name'] == protein_name]

        embeddings = []
        for idx in indices:
            residue_row = protein_data[protein_data['ResID'] == idx]
            if not residue_row.empty:
                embedding = residue_row.iloc[0, -321:-1].values
                embeddings.append(embedding)

        if len(embeddings) == 3:
            patch_embedding_sum = np.sum(embeddings, axis=0)
            patch_summary = {
                'Protein_Name': protein_name,
                'Patch_Indices': indices,
                'Status': 'positive',
                'Fold': fold
            }

            for i, value in enumerate(patch_embedding_sum):
                patch_summary[f'Embedding_{i+1}'] = value

            all_patch_summaries.append(patch_summary)

summary_df = pd.DataFrame(all_patch_summaries)
summary_df.to_csv('/content/drive/MyDrive/@antigen_epitope/abdb_new_combine/combined_positive_patches.csv', index=False)

print("Combined dataset shape:", summary_df.shape)

In [ ]:
summary_df

In [ ]:
import os
import pandas as pd
import numpy as np

base_model_dir = "/content/drive/MyDrive/@antigen_epitope/abdb_v4"
new_base_model_dir = "/content/drive/MyDrive/@antigen_epitope/abdb_new_method/"
all_patch_summaries = []

for fold in range(1, 6):
    txt_files_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    filtered_data_path = os.path.join(new_base_model_dir, f'fold_{fold}/data/all_data_with_SI.csv')

    filtered_combined_df = pd.read_csv(filtered_data_path)
    filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

    for i in range(1, 5):
        input_file_path = os.path.join(txt_files_dir, f'cnn_negative_patch_3points_part{i}.txt')

        with open(input_file_path, 'r') as file:
            lines = file.readlines()
            current_protein_name = None

            for line in lines:
                line = line.strip()
                if line.startswith('Patches for protein'):
                    current_protein_name = line.split(' ')[3][:-1]
                elif line.startswith('indices:'):
                    indices = line.split('[')[1].split(']')[0].split(', ')
                    indices = [int(idx) for idx in indices]

                    protein_data = filtered_combined_df[filtered_combined_df['name'] == current_protein_name]

                    embeddings = []
                    for idx in indices:
                        residue_row = protein_data[protein_data['ResID'] == idx]
                        if not residue_row.empty:
                            embedding = residue_row.iloc[0, -321:-1].values
                            embeddings.append(embedding)

                    if len(embeddings) == len(indices):
                        patch_embedding_sum = np.sum(embeddings, axis=0)
                        patch_summary = {
                            'Protein_Name': current_protein_name,
                            'Patch_Indices': tuple(indices),
                            'Status': 'negative'
                        }
                        for j, value in enumerate(patch_embedding_sum):
                            patch_summary[f'Embedding_{j+1}'] = value
                        all_patch_summaries.append(patch_summary)

summary_df_neg = pd.DataFrame(all_patch_summaries)
summary_df_neg.drop_duplicates(subset=['Protein_Name', 'Patch_Indices'], inplace=True)

output_path = '/content/drive/MyDrive/@antigen_epitope/abdb_new_combine/combined_negative_patches.csv'
summary_df_neg.to_csv(output_path, index=False)

print("Final negative patch shape:", summary_df_neg.shape)

In [ ]:
summary_df_neg

In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.utils import shuffle
import matplotlib.pyplot as plt

embedding_columns = [f'Embedding_{i}' for i in range(1, 321)]
combined_summary_df = pd.concat([summary_df, summary_df_neg], ignore_index=True)
X = combined_summary_df[embedding_columns].to_numpy()
y = combined_summary_df['Status'].replace({'positive': 1, 'negative': 0}).to_numpy()

X, y = shuffle(X, y, random_state=42)

print(f"Training data shape: {X.shape}, Labels shape: {y.shape}")

model = Sequential([
    Dense(128, activation='relu', input_shape=(X.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(
    X, y,
    epochs=100,
    batch_size=32,
    verbose=1
)

output_dir = './drive/MyDrive/@antigen_epitope/abdb_new_combine/'
os.makedirs(output_dir, exist_ok=True)
model_save_path = os.path.join(output_dir, 'classification_model_full_data.h5')
model.save(model_save_path)
print(f"Model saved to {model_save_path}")

plt.figure(figsize=(12, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.title('Training Loss and Accuracy')
plt.xlabel('Epochs')
plt.legend()
plt.tight_layout()
plt.show()